In [ ]:
%pip install pandas numpy

In [ ]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import CategoricalNB 
from sklearn.metrics import confusion_matrix, accuracy_score

# Naïve Bayes

Основывается на формуле Байеса (Расчет условной вероятности),\
которая используется, когда известна P( B | A ), а нужно узнать P( A | B )
$$ P(A|B) = \frac{P(B | A) * P(A)}{P(B)}$$


Почему "Наивный" байес? При работе с данными мы производим допущение, что все признаки независимы друг от друга (Ну или, если быть более точным, предполагаем, что их зависимость пренебрежимо мала)

## Работа алгоритма

### - Fit -

Строится частотная таблица / словарь (Каждому значению фактора сопостовляются условные вероятности его встретить при той или иной целевой переменной). После чего применяется сглаживание Лапласа:

$$P(Значение_{фактора} \mid Значение_{целевой}) = \frac{count(Значение_{фактора}, Значение_{целевой}) + \alpha}{Общее\_количество\_наборов\_с\_такой\_целевой + \alpha \cdot V}$$
$$P(Значения_{целевой}) = \frac{Количество объектов Значения_{целевой}}{Количество объектов в датасете}$$, где α - коэффициент сглаживания (Обычно единица),\
   а V - размер словаря (То есть количество различных значений, которые принимает фактор в датасете) \
   \
   \
   Сглаживание Лапласа нужно для избавления от логарифмирования нуля в дальнейшем, а используется именно оно, а не обычное прибавление единицы для того, чтобы после преобразования не нарушилось условие ```сумма всех возможных значений признака равна единице```
\
\
Далее логарифмируют посчитанные априорные вероятности для целевой и условные вероятности для значений признаков вероятности - делается это лишь для того, чтобы при последующем перемножении не вышло слишком маленькое число, которое float просто округлит в ноль
\
\
\
Обычно используют натуральный логарифм, но, в целом, разницы в основании особой нет, тк 
$$\log_b(x) = \frac{ln(x)}{ln(b)}$$
то есть при замене основания у нас просто произойдет домножение на $$ \frac{1}{ln(b)} $$ в следствие чего ответ модели не поменяется, тк домножение на константу неравенства не меняет его решения
\
\
\
После обучения сложностью ```O(n)``` выходит следующий результат:
- Список с априорными вероятностями целевой переменной {```P(Целевая=1)```, ```P(Целевая=2)```, ... , ```P(Целевая=m)```}
- Для каждого признака строится таблица (В памяти часто хранится как словарь словарей словарей по типу ключ-признак к значению, у которого ключ-значение_фактора к значению, у которого ключ-значение_целевой к значению логарифма условной вероятности значения признака при известном значениии целевой (В том же Sklearn тоже так хранится, но поскольку они ожидают по каждому столбцу последовательность от 0 до k значений, они это оптимизируют как 3 массива, где соответствующие вероятности по нужным индексам лежат ))
Пример таблички для признака (Для каждого признака хранится такая таблица)

| Значение признака | Вероятность при значении целевой = 0 | Вероятность при значении целевой = 1 | ... | Вероятность при значении целевой = m|
|:-:|:-:|:-:|:-:|:-:|
| $$Значение_1$$ | $$ln(P(Значение_1 \mid Целевая=0))$$|  $$ln(P(Значение_1 \mid Целевая=1))$$ | ...| ln(P(Значение_1 \mid Целевая=m)) |
| $$Значение_2$$ | $$ln(P(Значение_2 \mid Целевая=0))$$|  $$ln(P(Значение_2 \mid Целевая=1))$$ |... | ln(P(Значение_2 \mid Целевая=m))|
| ... | ... | ... | ... | ... |
| $$Значение_k$$ | $$ln(P(Значение_k \mid Целевая=0))$$|  $$ln(P(Значение_k \mid Целевая=1))$$ |...| ln(P(Значение_k \mid Целевая=m)) |

### - Inference -
Пример показывается при ``` количестве значений целевой переменной = 2 ``` для простоты восприятия, если переменных больше, то для каждого значения целевой переменной делаем расчеты вероятности, а потом ищем то значение, которое имеет максимальную вероятность - она и будет победителем 

Считаются условные вероятности:\
Для y = 1:
$$ P(y = 1 \mid X_{1 \dots n}) = \frac{P(y = 1) \cdot \prod_{i=1}^{n} P(x_i \mid y = 1)}{P(X_1, \dots, X_n)} $$
Для y = 0:
$$ P(y = 0 \mid X_{1 \dots n}) = \frac{P(y = 0) \cdot \prod_{i=1}^{n} P(x_i \mid y = 0)}{P(X_1, \dots, X_n)} $$

Теперь сравниваем
$$ P(y = 0 \mid X_{1 \dots n}) \ \lessgtr \ P(y = 1 \mid X_{1 \dots n}) $$
$$ \frac{P(y = 0) \cdot \prod_{i=1}^{n} P(x_i \mid y = 0)}{P(X_1, \dots, X_n)} \ \lessgtr \ \frac{P(y = 1) \cdot \prod_{i=1}^{n} P(x_i \mid y = 1)}{P(X_1, \dots, X_n)} $$
Очевидно, знаменатели равны и ненулевые (Тк иначе мы бы никогда не встретили исследуемый набор) => домножим на него
$$ P(y = 0) \cdot \prod_{i=1}^{n} P(x_i \mid y = 0)  \ \lessgtr \ P(y = 1) \cdot \prod_{i=1}^{n} P(x_i \mid y = 1) $$
Вспомним про округление float и что может произведение значений, меньших единицы, округлиться в ноль и прологорифмируем
$$ ln(P(y = 0) \cdot \prod_{i=1}^{n} P(x_i \mid y = 0)) \ \lessgtr \ ln(P(y = 1) \cdot \prod_{i=1}^{n} P(x_i \mid y = 1)) $$
Дальше вспомним, что логарифм произведения равняется сумме логарифмов множителей
$$ ln(P(y = 0)) + \sum_{i=1}^{n} ln(P(x_i \mid y = 0)) \ \lessgtr \ ln(P(y = 1)) + \sum_{i=1}^{n} ln(P(x_i \mid y = 1)) $$

Все необходимые данные для решения неравенства были посчитаны на этапе обучения, считаем ```y=1``` и ```y=0``` и даем ответ, что больше - то и будет ответом нашей модели

# Naïve Bayes (Laboratory)
##  In this assignment you will train a Naïve Bayes classifier on categorical data and predict individuals’ incomes.  Import the nbtrain.csv file.  Use the first 9010 records as training data and the remaining 1000 records as testing data.

### Подготовка данных:

In [ ]:
raw = pd.read_csv("resources/6-lab/nbtrain.csv")
raw

Преобразуем данные в численный тип, где каждое число представляет собой категорию

In [ ]:
tti = { # text to integer
    'age': {'GT 45': 0, '31-45': 1, '20-30': 2},
    'sex': {'M': 0, 'F': 1 },
    'educ': {'College': 0, 'Others': 1, 'Prof/Phd': 2},
    'income': {'10-50K': 0, '50-80K': 1, 'GT 80K': 2}
}

df = raw.replace(tti).astype(np.int8)
df

In [ ]:
train_df = df.iloc[:9010]
test_df = df.iloc[9010:]

Y_train = train_df['income']
X_train = train_df.drop(columns=['income'])

Y_test = test_df['income']
X_test = test_df.drop(columns=['income'])

# Показываем, что данные распределены в соответствии с заданием
len(train_df) == 9010 and len(test_df) == 1000

### a.      Construct the Naïve Bayes classifier from the training data, according to the formula “income ~ age + sex + educ”.  To do this, use the “naiveBayes” function from the “e1071” package.  Provide the model’s a priori and conditional probabilities.

In [ ]:
nb = CategoricalNB(alpha=1.0)
nb.fit(X_train, Y_train)

# Provide the model’s a priori and conditional probabilities
print(f"column names:        {nb.feature_names_in_}")
print(f"P(income) is {np.exp(nb.class_log_prior_)}") # Априорная вероятность
print(f"\nP(age | income) is:\n{nb.feature_log_prob_[0]}")
print(f"\nP(sex | income) is:\n{nb.feature_log_prob_[1]}")
print(f"\nP(educ | income) is:\n{nb.feature_log_prob_[2]}\n")

### b.      Score the model with the testing data and create the model’s confusion matrix.  Also, calculate the overall, 10-50K, 50-80K, and GT 80K misclassification rates. Explain the variation in the model’s predictive power across income classes.

Видно, что модель предсказывает в подавляющем большинстве класс ``` 0: '10-50K'```

Из-за того, что априорная вероятность равняется 0.8 для нулевого класса, он вечно "перетягивает одеяло на себя" и получает самую высокую вероятность, в следствие чего у нас большие ошибки на всех наборах, за исключением тех, где целевая переменная равна нулю

In [ ]:
Y_predicted = nb.predict(X_test)

# create the model’s confusion matrix
cm = confusion_matrix(Y_test, Y_predicted)

print(f"10-50K (0) составляют {len(Y_train[Y_train == 0]) / 90.1 :.1f}% от всего датасета")
print(f"50-80K (1) составляют {len(Y_train[Y_train == 1]) / 90.1 :.1f}% от всего датасета")
print(f"GT 80K (2) составляют {len(Y_train[Y_train == 2]) / 90.1 :.1f}% от всего датасета\n")

display(pd.DataFrame(cm, 
    index=['real_0', 'real_1', 'real_2'],
    columns=['pred_0', 'pred_1', 'pred_2']
))

print(f"misclassification rate for overall is {1 - sum(cm[i][i] for i in range(3)) / sum([sum(cm[i]) for i in range(3)]):.3f}")
print(f"misclassification rate for 10-50K (0) is {1 - cm[0][0] / sum(cm[0]):.3f}")
print(f"misclassification rate for 50-80K (1) is {1 - cm[1][1] / sum(cm[1]):.3f}")
print(f"misclassification rate for GT 80K (2) is {1 - cm[2][2] / sum(cm[2]):.3f}")

## 4.6        (Naïve Bayes)  As in assignment 4.5, import the nbtrain.csv file.  Use the first 9010 records as training data and the remaining 1000 records as testing data.

Уже сделано

### a.      Construct the classifier according to the formula “sex ~ age + educ + income”, and calculate the overall, female, and male misclassification rates.  Explain the misclassification rates?

Модель снова неадекватна и косит в сторону мужчин
Хотя целевая распределена приемлимо (56% против 43%), проблема кроется в посчитанных условных вероятностях, часто score женщин и мужчин относительно равен и потому условная вероятность мужчин просто перебивает условную вероятность женщин
Таким образом, хотя распределение и относительно одинакого, из-за почти равных score чаще выигрывают мужчины

А вот по income у женщин лучше положение и потому при условных заработках выше 80к модель будет отдавать предпочтение Female

In [ ]:
Y_train = train_df['sex']
X_train = train_df.drop(columns=['sex'])

Y_test = test_df['sex']
X_test = test_df.drop(columns=['sex'])

# Показываем, что данные распределены в соответствии с заданием
len(train_df) == 9010 and len(test_df) == 1000

In [ ]:
nb = CategoricalNB(alpha=1.0)
nb.fit(X_train, Y_train)

# Provide the model’s a priori and conditional probabilities
print(f"column names:        {nb.feature_names_in_}")
print(f"P(sex) is {np.exp(nb.class_log_prior_)}") # Априорная вероятность
print(f"\nP(age | sex) is:\n{nb.feature_log_prob_[0]}")
print(f"\nP(educ | sex) is:\n{nb.feature_log_prob_[1]}")
print(f"\nP(income | sex) is:\n{nb.feature_log_prob_[2]}\n")

In [ ]:
Y_predicted = nb.predict(X_test)

# create the model’s confusion matrix
cm = confusion_matrix(Y_test, Y_predicted)

print(f"Male (0) составляют {len(Y_train[Y_train == 0]) / 90.1 :.1f}% от всего датасета")
print(f"Female (1) составляют {len(Y_train[Y_train == 1]) / 90.1 :.1f}% от всего датасета")

display(pd.DataFrame(cm, 
    index=['real_0', 'real_1'],
    columns=['pred_0', 'pred_1']
))

print(f"misclassification rate for overall is {1 - sum(cm[i][i] for i in range(2)) / sum([sum(cm[i]) for i in range(2)]):.3f}")
print(f"misclassification rate for Male (0) is {1 - cm[0][0] / sum(cm[0]):.3f}")
print(f"misclassification rate for Female (1) is {1 - cm[1][1] / sum(cm[1]):.3f}")

### b.      Divide the training data into two partitions, according to sex, and randomly select 3500 records from each partition.  Reconstruct the model from part (a) from these 7000 records.  Provide the model’s a priori and conditional probabilities.

In [ ]:
train_male = train_df[train_df['sex'] == 0]
train_female = train_df[train_df['sex'] == 1]

sub_male = train_male.sample(n=3500, random_state=42)
sub_female = train_female.sample(n=3500, random_state=42)

train_data = pd.concat([sub_male, sub_female])
Y_tr = train_data['sex']
X_tr = train_data.drop(columns=['sex'])

nb = CategoricalNB(alpha=1.0)
nb.fit(X_tr, Y_tr)

# Provide the model’s a priori and conditional probabilities
print(f"column names:        {nb.feature_names_in_}")
print(f"P(sex) is {np.exp(nb.class_log_prior_)}") # Априорная вероятность
print(f"\nP(age | sex) is:\n{nb.feature_log_prob_[0]}")
print(f"\nP(educ | sex) is:\n{nb.feature_log_prob_[1]}")
print(f"\nP(income | sex) is:\n{nb.feature_log_prob_[2]}\n")

In [ ]:
Y_predicted = nb.predict(X_test)

# create the model’s confusion matrix
cm = confusion_matrix(Y_test, Y_predicted)

print(f"Male (0) составляют {len(Y_tr[Y_tr == 0]) / 70 :.1f}% от всего датасета")
print(f"Female (1) составляют {len(Y_tr[Y_tr == 1]) / 70 :.1f}% от всего датасета")

display(pd.DataFrame(cm, 
    index=['real_0', 'real_1'],
    columns=['pred_0', 'pred_1']
))

print(f"misclassification rate for overall is {1 - sum(cm[i][i] for i in range(2)) / sum([sum(cm[i]) for i in range(2)]):.3f}")
print(f"misclassification rate for Male (0) is {1 - cm[0][0] / sum(cm[0]):.3f}")
print(f"misclassification rate for Female (1) is {1 - cm[1][1] / sum(cm[1]):.3f}")

print(f"\nCount quantity of 0 in income: {(X_tr['income']==0).sum()}")

### c.       How well does the model classify the testing data?  Explain why.

Теперь перекос в сторону женщин, это случается, потому что у мужчин на ```0``` доход штраф в -0.3 score (что дает женщинам выигрыш на 0.18), по остальным факторам score у М и Ж относительно одинаков, в следствие чего income и оказывается решающим

### d.      Repeat step (b) 4 several times.  What effect does the random selection of records have on the model’s performance?

In [ ]:
def step_b(i):
    train_male = train_df[train_df['sex'] == 0]
    train_female = train_df[train_df['sex'] == 1]
    
    sub_male = train_male.sample(n=3500, random_state=i)
    sub_female = train_female.sample(n=3500, random_state=i)
    
    train_data = pd.concat([sub_male, sub_female])
    Y_tr = train_data['sex']
    X_tr = train_data.drop(columns=['sex'])
    
    nb = CategoricalNB(alpha=1.0)
    nb.fit(X_tr, Y_tr)
    
    Y_predicted = nb.predict(X_test)

    # create the model’s confusion matrix
    cm = confusion_matrix(Y_test, Y_predicted)

    pd.DataFrame(cm, 
        index=['real_0', 'real_1'],
        columns=['pred_0', 'pred_1']
    )

    return nb, cm, sub_male.index[1]

In [ ]:
cm_array = []
indx_set = set()

for i in range(100):
    nb, cm, indx = step_b(i)
    cm_array.append(cm)
    indx_set.add(indx)
 
cm_array = np.array(cm_array)

cm_mean = np.mean(cm_array, axis=0)

# Заметим, что матрица ошибок всегда одна и та же 
# (И совпадает с матрицей ошибок с шага b
display(cm_mean)

# Проверка, что каждый раз разные наборы объектов подавались на обучение
display(len(indx_set) > 95)

# Проверяем, действительно ли каждый раз одно и то же распределение
bool(np.all(cm_array == cm_array[0]))

### e.      What conclusions can one draw from this exercise?

Количество различных наборов достаточно мало: 3 * 2 * 3 * 3 = 54
Потому выборка в 7000 значений будет удовлетворять закону больших чисел - когда при достаточно большой выборке реальное среднее приближается к математическому ожиданию
![](https://thumb.wikimedia.org/wikipedia/commons/thumb/c/c9/Lawoflargenumbers.svg/1280px-Lawoflargenumbers.svg.png?utm_source=ru.wikipedia.org&utm_campaign=imageinfo&utm_content=thumbnail)

Потому при каждом случайном выборе новых 7000 значений получается одна и та же матрица ошибок